<a href="https://colab.research.google.com/github/eom-hope/ABCD-A/blob/20250414/yolo_V8_VS_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
from google.colab import files
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import io


In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
print("이미지를 업로드해주세요:")
uploaded = files.upload()


In [ ]:
def visualize_results(img, results, filename):
    # 원본 이미지 복사
    img_result = img.copy()

    # 결과 테이블 데이터 준비
    detection_data = []

    # 결과가 있으면 처리
    if len(results[0].boxes) > 0:
        boxes = results[0].boxes
        for i, box in enumerate(boxes):
            # 좌표와 클래스 정보 가져오기
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = model.names[cls_id]

            # 테이블 데이터에 추가
            detection_data.append({
                "번호": i+1,
                "클래스": cls_name,
                "신뢰도": f"{conf:.2f}",
                "위치(x1, y1, x2, y2)": f"({x1}, {y1}, {x2}, {y2})"
            })

            # 박스 그리기
            color = (0, 255, 0)  # 초록색 (BGR)
            cv2.rectangle(img_result, (x1, y1), (x2, y2), color, 2)

            # 텍스트 표시
            label = f"{cls_name}: {conf:.2f}"
            (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(img_result, (x1, y1-text_height-10), (x1+text_width, y1), color, -1)
            cv2.putText(img_result, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    # 결과 이미지 표시
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'YOLOv8 Detection Results - {filename}')
    plt.tight_layout()
    plt.show()

    # 분류 결과 테이블 표시
    if detection_data:
        df = pd.DataFrame(detection_data)
        print(f"\n{filename}에서 감지된 객체 ({len(detection_data)}개):")
        display(df)

        # 클래스별 통계
        class_counts = df['클래스'].value_counts().reset_index()
        class_counts.columns = ['클래스', '개수']

        print("\n클래스별 감지된 객체 수:")
        display(class_counts)

        # 클래스별 분포 차트
        plt.figure(figsize=(10, 6))
        plt.bar(class_counts['클래스'], class_counts['개수'], color='skyblue')
        plt.title("감지된 객체 클래스별 분포")
        plt.xlabel("클래스")
        plt.ylabel("개수")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print(f"\n{filename}에서 감지된 객체가 없습니다.")


In [ ]:
for filename in uploaded.keys():
    # 이미지 읽기
    img = cv2.imread(filename)
    if img is None:
        print(f"이미지를 열 수 없습니다: {filename}")
        continue

    # 객체 감지 수행
    results = model(img)

    # 결과 시각화
    visualize_results(img, results, filename)

    # 신뢰도 분포 히스토그램 (감지된 객체가 있을 경우)
    if len(results[0].boxes) > 0:
        confidences = [float(box.conf[0]) for box in results[0].boxes]

        plt.figure(figsize=(8, 5))
        plt.hist(confidences, bins=10, range=(0, 1), alpha=0.7, color='blue')
        plt.title("감지된 객체의 신뢰도 분포")
        plt.xlabel("신뢰도 점수")
        plt.ylabel("객체 수")
        plt.grid(alpha=0.3)
        plt.show()

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics opencv-python-headless

# 필요한 라이브러리 import
import cv2
import numpy as np
import os
import zipfile
import glob
from ultralytics import YOLO
import matplotlib.pyplot as plt
from google.colab import files
from tqdm.notebook import tqdm
import shutil
from IPython.display import display, clear_output

# YOLOv8 모델 로드
print("YOLOv8 모델 로딩 중...")
model = YOLO('yolov8n.pt')  # 'n'은 nano 모델, 다른 옵션: 's', 'm', 'l', 'x'
print("모델 로딩 완료!")

# 지원하는 동영상 확장자
VIDEO_EXTENSIONS = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm']

# 파일 업로드 및 동영상 추출 함수
def extract_videos_from_file():
    print("파일을 업로드해주세요 (ZIP 또는 동영상 파일):")
    uploaded = files.upload()

    if not uploaded:
        print("파일이 업로드되지 않았습니다.")
        return []

    # 임시 작업 디렉토리 생성
    temp_dir = "temp_extracted_videos"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)

    video_files = []

    # 업로드된 각 파일 처리
    for filename in uploaded.keys():
        file_path = filename
        file_ext = os.path.splitext(filename)[1].lower()

        # ZIP 파일 처리
        if file_ext == '.zip':
            print(f"\nZIP 파일 '{filename}' 압축 해제 중...")

            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                # ZIP 파일 내용 목록 출력
                file_list = zip_ref.namelist()
                print(f"ZIP 파일 내 항목 수: {len(file_list)}")

                # ZIP 파일 압축 해제
                zip_ref.extractall(temp_dir)

                # 압축 해제된 폴더에서 동영상 파일 찾기
                for ext in VIDEO_EXTENSIONS:
                    found_videos = glob.glob(os.path.join(temp_dir, '**', f'*{ext}'), recursive=True)
                    video_files.extend(found_videos)

                print(f"추출된 동영상 파일 수: {len(video_files)}")

        # 동영상 파일 처리
        elif file_ext in VIDEO_EXTENSIONS:
            # 파일을 임시 디렉토리로 복사
            dest_path = os.path.join(temp_dir, filename)
            shutil.copy(file_path, dest_path)
            video_files.append(dest_path)
            print(f"\n동영상 파일 감지됨: {filename}")

        else:
            print(f"\n'{filename}'은(는) 지원되지 않는 파일 형식입니다.")

    # 발견된 동영상 파일 목록 출력
    if video_files:
        print("\n처리할 동영상 파일 목록:")
        for i, video_path in enumerate(video_files):
            print(f"{i+1}. {os.path.basename(video_path)}")
    else:
        print("\n처리할 동영상 파일이 없습니다.")

    return video_files

# 동영상 처리 함수
def process_video(video_path, conf_threshold=0.25, sample_rate=2):
    print(f"\n동영상 처리 중: {os.path.basename(video_path)}")

    # 동영상 열기
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"동영상 파일을 열 수 없습니다: {video_path}")
        return None

    # 동영상 정보 가져오기
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    print(f"\n동영상 정보:")
    print(f"- 해상도: {width}x{height} 픽셀")
    print(f"- FPS: {fps:.2f}")
    print(f"- 총 프레임 수: {total_frames}")
    print(f"- 재생 시간: {duration:.2f}초 ({duration/60:.2f}분)")

    # 결과 저장 설정
    output_dir = "yolo_results"
    os.makedirs(output_dir, exist_ok=True)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_path = os.path.join(output_dir, f"{video_name}_detected.mp4")

    # 출력 동영상 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # 결과 저장용 변수
    class_counts = {}  # 클래스별 객체 카운트
    frame_objects = {}  # 프레임별 객체 수

    # 프로그레스 바
    pbar = tqdm(total=total_frames)

    # 동영상 처리 시작
    frame_count = 0
    processed_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        pbar.update(1)

        # 지정된 간격으로 프레임 처리
        if frame_count % sample_rate == 0:
            processed_count += 1

            # 객체 감지 수행
            results = model(frame, conf=conf_threshold)

            # 결과 시각화
            annotated_frame = results[0].plot()

            # 감지된 객체 카운트
            boxes = results[0].boxes
            detected_objects = len(boxes)
            frame_objects[frame_count] = detected_objects

            # 클래스별 객체 카운트
            for box in boxes:
                cls_id = int(box.cls[0])
                cls_name = model.names[cls_id]
                if cls_name in class_counts:
                    class_counts[cls_name] += 1
                else:
                    class_counts[cls_name] = 1

            # 프레임 정보 추가
            progress = frame_count / total_frames * 100
            cv2.putText(annotated_frame, f"Frame: {frame_count}/{total_frames} ({progress:.1f}%)",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            cv2.putText(annotated_frame, f"Objects: {detected_objects}",
                       (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # 처리 중인 프레임 미리보기 (10% 간격으로)
            if processed_count <= 3 or frame_count % int(total_frames/10) == 0:
                plt.figure(figsize=(10, 6))
                plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
                plt.title(f'처리 중: {frame_count}/{total_frames} 프레임 ({progress:.1f}%)')
                plt.axis('off')
                plt.show()

            # 결과 저장
            out.write(annotated_frame)
        else:
            # 처리하지 않는 프레임은 원본 그대로 저장
            out.write(frame)

    # 자원 해제
    cap.release()
    out.release()
    pbar.close()

    # 결과 통계 및 시각화
    print(f"\n동영상 처리 완료: {os.path.basename(video_path)}")
    print(f"- 총 프레임: {total_frames}")
    print(f"- 처리된 프레임: {processed_count}")

    # 클래스별 객체 수 표시
    if class_counts:
        print("\n감지된 객체 통계:")
        sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
        for cls_name, count in sorted_classes:
            print(f"- {cls_name}: {count}개")

        # 상위 10개 클래스 시각화
        top_classes = dict(sorted_classes[:10])
        plt.figure(figsize=(12, 6))
        plt.bar(top_classes.keys(), top_classes.values(), color='skyblue')
        plt.title(f"{os.path.basename(video_path)} - 상위 감지된 객체")
        plt.xlabel("클래스")
        plt.ylabel("객체 수")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("\n감지된 객체가 없습니다.")

    return output_path

# 메인 처리 함수
def main():
    print("=" * 60)
    print("파일에서 동영상 추출 후 YOLOv8로 객체 식별")
    print("=" * 60)

    # 1. 파일 업로드 및 동영상 추출
    video_files = extract_videos_from_file()

    if not video_files:
        print("처리할 동영상 파일이 없습니다.")
        return

    # 2. 처리 옵션 설정
    try:
        conf_threshold = float(input("\n감지 신뢰도 임계값 (0.1~0.9, 기본값 0.25): ") or "0.25")
        conf_threshold = max(0.1, min(0.9, conf_threshold))

        sample_rate = int(input("처리할 프레임 간격 (1: 모든 프레임, 2: 2프레임마다, 기본값 2): ") or "2")
        sample_rate = max(1, sample_rate)
    except ValueError:
        print("올바른 값을 입력하지 않았습니다. 기본값을 사용합니다.")
        conf_threshold = 0.25
        sample_rate = 2

    print(f"\n설정된 옵션:")
    print(f"- 감지 신뢰도 임계값: {conf_threshold}")
    print(f"- 프레임 처리 간격: {sample_rate}프레임마다 처리")

    # 3. 각 동영상 처리
    processed_videos = []
    for video_path in video_files:
        output_path = process_video(video_path, conf_threshold, sample_rate)
        if output_path:
            processed_videos.append(output_path)

    # 4. 처리 완료 및 결과 다운로드
    if processed_videos:
        print("\n모든 동영상 처리 완료!")
        print(f"처리된 동영상 수: {len(processed_videos)}")

        # 결과 동영상 다운로드
        for output_path in processed_videos:
            if os.path.exists(output_path):
                print(f"\n결과 동영상 다운로드 중: {os.path.basename(output_path)}")
                files.download(output_path)
    else:
        print("\n처리된 동영상이 없습니다.")

    # 5. 임시 파일 정리
    try:
        if os.path.exists("temp_extracted_videos"):
            shutil.rmtree("temp_extracted_videos")
    except Exception as e:
        print(f"임시 파일 정리 중 오류 발생: {e}")

# 실행
if __name__ == "__main__":
    main()